# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR⁲ dataset using the `mlcroissant` library. The dataset contains ordered logistic regression outputs and survey metadata relating to adoption predictors for indigenous and modern knowledge in rangeland management.

### Dataset Source
The dataset is defined by a Croissant schema linked below.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata fields
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will enumerate the available record sets (`@id`), fields (`@id`) and columns in the dataset. All references will use the `@id` for consistency with the Croissant schema paradigm.

In [ ]:
# List all record sets in the dataset
print("Available record sets and their field IDs:")
record_sets = []
for rs in dataset.record_sets:
    print(f"  RecordSet @id: {rs['@id']}")
    record_sets.append(rs['@id'])
    # List the fields of the record set
    if 'field' in rs:
        for field in rs['field']:
            if isinstance(field, dict) and '@id' in field:
                print(f"    - field @id: {field['@id']}")
            elif isinstance(field, str):
                print(f"    - field @id: {field}")
    # List the columns, if any
    if 'column' in rs:
        for col in rs['column']:
            if isinstance(col, dict) and '@id' in col:
                print(f"    - column @id: {col['@id']}")
            elif isinstance(col, str):
                print(f"    - column @id: {col}")

if not record_sets:
    print("  No record sets found in this Croissant schema. Try loading by distribution if present.")

# For this dataset, we may need to inspect 'distribution' as record sets list appears empty.
if hasattr(metadata, 'distribution') and metadata.distribution:
    print("\nAvailable distributions and their @id:")
    for dist in metadata.distribution:
        if isinstance(dist, dict) and '@id' in dist:
            print(f"  - Distribution @id: {dist['@id']}")
        elif isinstance(dist, str):
            print(f"  - Distribution @id: {dist}")

## 3. Data Extraction
Load data from a specific record set (via `@id`) or distribution if record sets are not defined.

We'll demonstrate loading records from each identified record set (or distribution) and constructing a DataFrame. All entity references are managed using their Croissant `@id`.

In [ ]:
dataframes = dict()

# If record sets are empty, load from available distributions
if not record_sets:
    # Extract distribution @ids
    distributions = []
    for dist in metadata.distribution:
        if isinstance(dist, dict) and '@id' in dist:
            distributions.append(dist['@id'])
        elif isinstance(dist, str):
            distributions.append(dist)
    print(f"Attempting to load records from distribution(s): {distributions}")
    # We'll attempt to load from each distribution; some may be datafile objects
    for dist_id in distributions:
        try:
            records_list = list(dataset.records(record_set=dist_id))
            if len(records_list) > 0:
                df = pd.DataFrame(records_list)
                dataframes[dist_id] = df
                print(f"Loaded {len(df)} records from distribution @id: {dist_id}")
            else:
                print(f"No records found for distribution @id: {dist_id}")
        except Exception as e:
            print(f"Could not load records for distribution @id: {dist_id}: {e}")
else:
    for rs_id in record_sets:
        try:
            records = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records from record set @id: {rs_id}")
        except Exception as e:
            print(f"Could not load records for record set @id: {rs_id}: {e}")

# Show columns of the first available DataFrame
if dataframes:
    first_id = list(dataframes.keys())[0]
    print(f"\nColumns for {first_id}:")
    print(dataframes[first_id].columns.tolist())
    dataframes[first_id].head()
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. We'll reference all fields by their `@id` (i.e., column name as loaded).

In [ ]:
# Select dataframe and inspect numeric fields
if dataframes:
    # Use the first loaded DataFrame for demonstration
    df_id = list(dataframes.keys())[0]
    df = dataframes[df_id]
    print(f"Data preview for @id: {df_id}")
    display(df.head())
    
    # Select the first numeric column for analysis, referenced by its @id
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    if not numeric_cols:
        print("No numeric fields found in the data.")
    else:
        numeric_field_id = numeric_cols[0]
        print(f"\nUsing numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() # Example: filter above mean
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Attempt grouping by a categorical field
        cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
        group_field = None
        if len(cat_cols) > 0:
            group_field = cat_cols[0]
            print(f"\nGrouping by categorical field: {group_field} (referenced by @id)")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of '{numeric_field_id}' by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No categorical fields found for grouping.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize distributions or relationships in the dataset. We'll use the first available numeric field and, if possible, a grouping field, always referencing columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution and boxplot by group if available
if dataframes:
    df_id = list(dataframes.keys())[0]
    df = dataframes[df_id]
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.tight_layout()
        plt.show()
        # Boxplot by first categorical field if exists
        cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
        if cat_cols:
            group_field_id = cat_cols[0]
            plt.figure(figsize=(10,4))
            sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
            plt.xticks(rotation=45, ha='right')
            plt.title(f"{numeric_field_id} by {group_field_id}")
            plt.xlabel(group_field_id)
            plt.ylabel(numeric_field_id)
            plt.tight_layout()
            plt.show()
else:
    print("No dataframes available for visualization.")

## 6. Conclusion
This notebook demonstrated end-to-end exploration of the FAIR⁲ dataset using the `mlcroissant` library, referencing all entities by their Croissant `@id`. We loaded the dataset schema, identified available record sets (or distributions), loaded records, performed data filtering and normalization, and visualized key attributes. The `mlcroissant` approach ensures consistent and reproducible access to FAIR data as published.

You can adapt this template to other datasets by updating the Croissant schema URL and adjusting analysis to your specific record set and field `@id`s as needed.